# Feature Engineering

## Team Selection

Choose the team, Grand Prix, and season that have already been processed. These parameters define the exact comparison window for the analysis.

In [ ]:
team = "Aston Martin"
grand_prix = "Japanese Grand Prix"
year = 2025

## Imports and Configuration

Load dependencies, configure the notebook path, and prepare the modules used in the exploration steps.

In [ ]:
import sys
import fastf1
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

from src.load_save_data import *
from src.preprocess_data import *
from src.explore_data import *
from src.feature_engineering import *

## Load Processed Data

Read the cached processed laps for both teammates so the notebook can work from already cleaned telemetry.

In [ ]:
session = fastf1.get_session(year, grand_prix, "Q")
session.load()

driver_1, driver_2 = get_drivers(session, team)
segment = get_segment(session, driver_1, driver_2)
turns = get_turns(session)

try:
    lap_1, lap_2 = load_cache(year, grand_prix, segment, driver_1, driver_2)
except Exception as e:
    print(f"Processed data for {year} {grand_prix} {driver_1} and {driver_2} not found: {e}")

## Compute Corner Features

In [ ]:
try:
    corners_1 = analyze_corners(lap_1["Telemetry"], turns)
    corners_2 = analyze_corners(lap_2["Telemetry"], turns)
except Exception as e:
    print(f"Corners analysis failed: {e}")
    
print(f"Corners data for {driver_1}:\n\n{corners_1}\n")
print(f"Corners data for {driver_2}:\n\n{corners_2}")

## Compute Braking features

In [ ]:
try:
    braking_zones_1 = analyze_braking_zones(lap_1["Telemetry"])
    braking_zones_2 = analyze_braking_zones(lap_2["Telemetry"])
except Exception as e:
    print(f"Braking zones analysis failed: {e}")

try:
    braking_zones_1 = assign_braking_zones_to_turns(braking_zones_1, turns)
    braking_zones_2 = assign_braking_zones_to_turns(braking_zones_2, turns)
except Exception as e:
    print(f"Assignment of braking zones to turns failed: {e}")
    
print(f"Braking zones for {driver_1}:\n\n: {braking_zones_1}\n")
print(f"Braking zones for {driver_2}:\n\n: {braking_zones_2}")

## Compute Acceleration features

In [ ]:
try:
    acceleration_zones_1 = analyze_acceleration_zones(lap_1["Telemetry"], braking_zones_1)
    acceleration_zones_2 = analyze_acceleration_zones(lap_2["Telemetry"], braking_zones_2)
except Exception as e:
    print(f"Acceleration zones analysis failed: {e}")
    
try:
    acceleration_zones_1 = analyze_throttle_lifts(lap_1["Telemetry"], acceleration_zones_1)
    acceleration_zones_2 = analyze_throttle_lifts(lap_2["Telemetry"], acceleration_zones_2)
except Exception as e:
    print(f"Throttle lifts analysis failed: {e}")

print(f"Acceleration zones for {driver_1}:\n\n: {acceleration_zones_1}\n")
print(f"Acceleration zones for {driver_2}:\n\n: {acceleration_zones_2}\n")

## Compute gear features

In [ ]:
try:
    gear_shifts_1 = analyze_gear_shifts(lap_1["Telemetry"])
    gear_shifts_2 = analyze_gear_shifts(lap_2["Telemetry"])
except Exception as e:
    print(f"Gear shifts analysis failed: {e}")

print(f"Gear shifts for {driver_1}:\n\n: {gear_shifts_1}\n")
print(f"Gear shifts for {driver_2}:\n\n: {gear_shifts_2}\n")

In [ ]:
lap_1["Corners"] = corners_1
lap_1["BrakingZones"] = braking_zones_1 
lap_1["AccelerationZones"] = acceleration_zones_1
lap_1["GearShifts"] = gear_shifts_1


lap_2["Corners"] = corners_2
lap_2["BrakingZones"] = braking_zones_2
lap_2["AccelerationZones"] = acceleration_zones_2
lap_2["GearShifts"] = gear_shifts_2

try:
    save_cache(year, grand_prix, segment, lap_1, lap_2)
except Exception as e:
    print(f"Failed to save lap cache: {e}")